# RIFT Week 9: generative NLL, Qwen2.5-1.5B-Instruct

Enable Internet; select GPU T4 x2. Two independent workers, one per GPU, not pooled VRAM.

Default `RUN_TRAINING=False`: setup, tiny CPU tests, dataset audit and job plan only. When ready, use smoke first, then development, then frozen confirmation. Development: 36 jobs; confirmation: 72 jobs. All declared seeds remain in the analysis even when a session runs only a subset.

Controls are `raw` (product addition before rank cap), `freshness`, and `alignfed_calibration` (whole-update gate), NOT full FedEx/AlignFed reproductions. Candidates: RIFT, RIFT-Diag and RIFT-Core. This matrix trains on short generative QA; it does not measure forgetting of old classification-trained checkpoints.


In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import json
import os
import signal
import subprocess
import sys
import time
import zipfile
from IPython.display import display, Markdown, FileLink

REPO_URL = "https://github.com/TrgPhan/VASTLoRA.git"
REPO_REF = "3f9c5b226298996419308d67d493bcf35f506d8b"
MODE = "development"  # smoke / development / confirmation
RUN_TRAINING = False
CONFIRM_PROTOCOL_FROZEN = False
GPU_IDS = [0, 1]
MAX_JOBS = None  # Limit NEW jobs this session, not the full statistical cohort.
RETRY_INCOMPLETE = False  # Archive an interrupted job and restart its seed from scratch.
RESUME_ROOTS = []  # Paths to extracted Week 9 v3 output roots under /kaggle/input.
if MODE not in {"smoke", "development", "confirmation"}:
    raise ValueError("Unknown MODE")
if MODE == "confirmation" and RUN_TRAINING and not CONFIRM_PROTOCOL_FROZEN:
    raise RuntimeError("Review development and freeze protocol before confirmation.")
WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / ("RIFTLoRA-week9-" + REPO_REF[:8])
OUTPUT_ROOT = WORK_ROOT / ("week9_v3_" + MODE)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print({"mode": MODE, "training": RUN_TRAINING, "output": str(OUTPUT_ROOT), "commit": REPO_REF})


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "checkout", "--detach", REPO_REF], cwd=REPO_DIR, check=True)
resolved = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"],
                                cwd=REPO_DIR, text=True).strip()
if resolved != REPO_REF or dirty:
    raise RuntimeError("Checkout differs from pinned clean implementation; use a fresh REPO_DIR.")
RUNNER = REPO_DIR / "scripts/run_week9_generation.py"
if not RUNNER.exists():
    raise RuntimeError("Week 9 runner is missing from the pinned GitHub commit.")

# Preserve Kaggle's CUDA Torch build. NF4 uses bitsandbytes, not optional torchao.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
constraints = WORK_ROOT / "week9-torch-constraint.txt"
constraints.write_text("torch==" + metadata.version("torch") + "\n", encoding="utf-8")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-c", str(constraints), "-e", ".[scale,dev,generation]",
    "transformers==5.10.2", "peft==0.20.0", "accelerate==1.14.0",
    "bitsandbytes==0.50.2", "datasets==5.0.0", "rouge-score==0.1.2",
], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "-c",
    "import torch, peft, transformers, bitsandbytes; "
    "print({'torch':torch.__version__, 'peft':peft.__version__, "
    "'transformers':transformers.__version__, 'bitsandbytes':bitsandbytes.__version__})"
], check=True)


In [ ]:
# Cache the actual pinned tokenizer; the following integration test must not skip.
tokenizer_config = json.loads((REPO_DIR / "configs/local_1_5b_rift_development.json").read_text())["model"]
subprocess.run([sys.executable, "-c",
    "import sys; from transformers import AutoTokenizer; AutoTokenizer.from_pretrained(sys.argv[1], revision=sys.argv[2])",
    tokenizer_config["name"], tokenizer_config["revision"]], check=True)
test_env = os.environ.copy()
test_env.update(CUDA_VISIBLE_DEVICES="", PYTEST_DISABLE_PLUGIN_AUTOLOAD="1", TOKENIZERS_PARALLELISM="false", REQUIRE_WEEK9_TOKENIZER="1")
subprocess.run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_week9_generation.py", "tests/test_week9_analysis.py", "tests/test_week9_launcher.py",
    "tests/test_week9_real_tokenizer.py",
    "tests/test_scale_objective.py", "tests/test_core_repair.py",
], cwd=REPO_DIR, env=test_env, check=True)

base_command = [sys.executable, "-u", str(RUNNER), "--output-root", str(OUTPUT_ROOT)]
base_command += ["--smoke"] if MODE == "smoke" else ["--phase", MODE]
subprocess.run(base_command + ["--dry-run"], cwd=REPO_DIR, check=True)
subprocess.run(base_command + ["--prepare-only"], cwd=REPO_DIR, check=True)
plan_command = base_command + ["--plan-only"]
for root in RESUME_ROOTS:
    plan_command += ["--resume-root", str(root)]
subprocess.run(plan_command, cwd=REPO_DIR, check=True)
matrix = json.loads((OUTPUT_ROOT / "matrix.json").read_text())
plan = json.loads((OUTPUT_ROOT / "job_plan.json").read_text())
expected = {"smoke": 6, "development": 36, "confirmation": 72}[MODE]
assert len(plan) == expected
display({"expected_jobs": expected, "methods": matrix["methods"], "seeds": matrix["seeds"],
         "regimes": [r["name"] for r in matrix["regimes"]], "primary_target": matrix["primary_target"],
         "eval_examples": matrix["tasks"][0]["eval_examples"], "eval_split": matrix["tasks"][0]["eval_split"]})
display(json.loads((OUTPUT_ROOT / "data_audit.json").read_text()))
environment = {"repo_commit": resolved, "python": sys.version,
               "packages": subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True).splitlines()}
(OUTPUT_ROOT / f"notebook_environment_{time.time_ns()}.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")


## Optional Training

Leave `RUN_TRAINING=False` to stop at preflight. For a short hardware check, set `MODE='smoke'` and enable training. Inspect generation-limit rate and predictions before running the development matrix. Full confirmation requires `CONFIRM_PROTOCOL_FROZEN=True` and a fresh output directory if the protocol changes.

Completed matching runs are skipped. `RESUME_ROOTS` imports only fully validated results from this exact implementation. Interrupted jobs are not checkpoint-resumable: `RETRY_INCOMPLETE=True` archives them and retrains their seed. Any OOM or worker error stops the launcher's other worker too; budgets are never reduced automatically.


In [ ]:
if RUN_TRAINING:
    import torch
    if not GPU_IDS or len(set(GPU_IDS)) != len(GPU_IDS) or any(g < 0 or g >= torch.cuda.device_count() for g in GPU_IDS):
        raise RuntimeError(f"Invalid GPU_IDS {GPU_IDS}; detected {torch.cuda.device_count()} GPUs.")
    gpu_info = subprocess.check_output(["nvidia-smi"], text=True)
    print(gpu_info)
    (OUTPUT_ROOT / "nvidia-smi.txt").write_text(gpu_info, encoding="utf-8")
    # Cache weights before parallel workers start; the preflight-only path does not download weights.
    from huggingface_hub import snapshot_download
    first_config = json.loads(Path(plan[0]["command"][plan[0]["command"].index("--config") + 1]).read_text())
    model = first_config["model"]
    snapshot_download(model["name"], revision=model["revision"])

    command = list(base_command)
    for gpu in GPU_IDS:
        command += ["--gpu", str(gpu)]
    for root in RESUME_ROOTS:
        command += ["--resume-root", str(root)]
    if MAX_JOBS is not None:
        command += ["--max-jobs", str(MAX_JOBS)]
    if RETRY_INCOMPLETE:
        command.append("--retry-incomplete")
    assert all(isinstance(arg, str) for arg in command)
    launcher_log = OUTPUT_ROOT / f"launcher_{time.time_ns()}.log"
    process = None
    with launcher_log.open("w", encoding="utf-8") as log:
        try:
            process = subprocess.Popen(command, cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT,
                                       start_new_session=True)
            print({"launcher_pid": process.pid, "log": str(launcher_log)})
            last_progress = 0
            while process.poll() is None:
                if time.monotonic() - last_progress > 30:
                    print(launcher_log.read_text(encoding="utf-8", errors="replace")[-3000:], flush=True)
                    last_progress = time.monotonic()
                time.sleep(2)
            if process.returncode:
                raise RuntimeError(f"Launcher failed ({process.returncode}); inspect {launcher_log}.")
        finally:
            if process is not None and process.poll() is None:
                os.killpg(process.pid, signal.SIGINT)
                try:
                    process.wait(timeout=45)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
    print(launcher_log.read_text(encoding="utf-8", errors="replace")[-5000:])
else:
    print("Preflight complete. No Qwen training jobs were started.")


In [ ]:
for target in ["rift_core", "rift_diag", "rift"]:
    subprocess.run([sys.executable, str(REPO_DIR / "scripts/analyze_week9_generation.py"),
                    "--input-dir", str(OUTPUT_ROOT), "--target", target], cwd=REPO_DIR, check=True)
report = OUTPUT_ROOT / "analysis_rift_core" / "results.md"
display(Markdown(report.read_text(encoding="utf-8")))
verdict = json.loads((OUTPUT_ROOT / "analysis_rift_core" / "verdict.json").read_text())
display({"status": verdict["status"], "completed": len(verdict["rows"]), "missing": len(verdict["missing"]),
         "issues": verdict["issues"], "warnings": verdict["warnings"]})
# This still exports partial runs after a failure when this cell is executed manually.
archive = WORK_ROOT / (OUTPUT_ROOT.name + ".zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file() and path.name != ".launcher.lock":
            z.write(path, path.relative_to(OUTPUT_ROOT.parent))
display(FileLink(str(archive)))


## Interpretation

Primary metric: response token NLL including EOS; perplexity is its exponential. Report paired seed differences against every declared control in each regime. A Week 9 NLL gate pass needs all 72 clean confirmation runs, six paired seeds, CI95 upper bound <= 0.05 nats/token, at least eight late events per run and target acceptance >= 0.5. The primary target is RIFT-Core; other target analyses are exploratory.

Also inspect monitor harmful/late harmful, ROUGE-L, exact match, generation-limit rate, runtime and memory. ROUGE is not factual accuracy. No classification accuracy is reported for this task. A Week 9 NLL gate pass is not a global thesis GO, and no late events means late harmful is unavailable, not zero-risk.
